# Phase 13 — ChromaDB Index Building (SQL + NoSQL)

Builds persistent vector indexes for both RAG corpora using the trained SAR models.
After this phase, the inference pipeline retrieves similar examples **instantly** from ChromaDB
instead of re-encoding all 7000+ corpus questions on every startup.

## What this notebook does

For each track (SQL and NoSQL):
1. Loads the trained `SchemaAwareModel` (from Google Drive)
2. Loads the RAG corpus (`spider_sql_rag.json` / `spider_nosql_rag.json`)
3. Encodes all corpus questions: BGE → SchemaAwareModel → normalized 1024-dim vector
4. Stores embeddings + metadata in a persistent ChromaDB collection
5. Saves the index to Google Drive

## Before vs After

| | Without ChromaDB (SARRetriever) | With ChromaDB (ChromaSARRetriever) |
|---|---|---|
| **Startup time** | ~30 sec (re-encodes all questions) | **Instant** (loads pre-built index) |
| **Query time** | ~10 ms (in-memory matmul) | ~10 ms (ChromaDB HNSW search) |
| **Persistence** | None — lost on restart | Permanent on Drive |
| **Memory** | Stores all embeddings in RAM | Disk-backed, small RAM footprint |

## Prerequisites
- Phase 12A complete: `sar_sql/sar_model.pt` on Drive ✅
- Phase 12B complete: `sar_nosql/sar_model.pt` on Drive ✅

> **No GPU needed.** BGE encoding and SchemaAwareModel forward pass run fine on CPU.
> Select **CPU** runtime or keep T4 — both work.

## Cell 1 — Mount Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# SAR model paths (from Phase 12A/12B)
SAR_SQL_MODEL   = '/content/drive/MyDrive/codegen/checkpoints/sar_sql/sar_model.pt'
SAR_NOSQL_MODEL = '/content/drive/MyDrive/codegen/checkpoints/sar_nosql/sar_model.pt'

# Output index paths
CHROMA_SQL_DIR   = '/content/drive/MyDrive/codegen/indexes/chroma_sql'
CHROMA_NOSQL_DIR = '/content/drive/MyDrive/codegen/indexes/chroma_nosql'

os.makedirs(CHROMA_SQL_DIR,   exist_ok=True)
os.makedirs(CHROMA_NOSQL_DIR, exist_ok=True)

# Verify models exist
for path, label in [(SAR_SQL_MODEL, 'SAR SQL'), (SAR_NOSQL_MODEL, 'SAR NoSQL')]:
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1e6
        print(f'{label} model: {size_mb:.1f} MB ✅')
    else:
        print(f'{label} model: NOT FOUND ❌  ({path})')

## Cell 2 — Clone or update repo

In [ ]:
%%bash
set -e
REPO_DIR="/content/Codegen"
BRANCH="phase/13-chroma-index"

if [ -d "$REPO_DIR/.git" ]; then
    cd "$REPO_DIR"
    git fetch origin
    git checkout $BRANCH
    git pull origin $BRANCH
else
    git clone https://github.com/kethansplunk/Codegen.git "$REPO_DIR"
    cd "$REPO_DIR"
    git checkout $BRANCH
fi

echo "Branch : $(git branch --show-current)"
echo "Commit : $(git log --oneline -1)"

## Cell 3 — Install dependencies

- **chromadb**: vector database with HNSW indexing and persistence
- **FlagEmbedding**: BGE model for question encoding

In [ ]:
!pip install -q chromadb FlagEmbedding scikit-learn
print('Dependencies installed ✅')

## Cell 4 — Build SQL ChromaDB index

**What happens:**
1. Loads `sar_sql/sar_model.pt` from Drive
2. Loads `spider_sql_rag.json` (7000 entries)
3. Encodes all 7000 questions: BGE (frozen) → SchemaAwareModel (trained) → L2-normalized
4. Stores embeddings + metadata in ChromaDB collection `sar_sql`
5. Saves to Drive at `indexes/chroma_sql/`

**Metadata stored per entry:**  `question`, `sql`, `db_name`, `structural_type` (JSON)

**Expected time:** ~3-5 minutes (BGE encoding is the slow part)

In [ ]:
%%bash
cd /content/Codegen

echo "=== Building SQL ChromaDB index ==="
python -m scripts.build_chroma_index \
    --corpus  Data/rag_corpus/spider_sql_rag.json \
    --model   /content/drive/MyDrive/codegen/checkpoints/sar_sql/sar_model.pt \
    --out     /content/drive/MyDrive/codegen/indexes/chroma_sql \
    --name    sar_sql

## Cell 5 — Build NoSQL ChromaDB index

Same process, NoSQL corpus (5697 entries) and NoSQL SAR model.

**Extra metadata stored:** `mql_pipeline` (JSON), `mql_collection`

**Expected time:** ~2-4 minutes

In [ ]:
%%bash
cd /content/Codegen

echo "=== Building NoSQL ChromaDB index ==="
python -m scripts.build_chroma_index \
    --corpus  Data/rag_corpus/spider_nosql_rag.json \
    --model   /content/drive/MyDrive/codegen/checkpoints/sar_nosql/sar_model.pt \
    --out     /content/drive/MyDrive/codegen/indexes/chroma_nosql \
    --name    sar_nosql

## Cell 6 — Verify indexes on Drive

In [ ]:
import os

for label, chroma_dir in [('SQL', CHROMA_SQL_DIR), ('NoSQL', CHROMA_NOSQL_DIR)]:
    print(f'\n=== {label} ChromaDB index ===')
    if not os.path.exists(chroma_dir):
        print('  ❌ Directory not found')
        continue
    total_mb = sum(
        os.path.getsize(os.path.join(dp, f))
        for dp, _, files in os.walk(chroma_dir)
        for f in files
    ) / 1e6
    files = list(os.walk(chroma_dir))
    print(f'  Total size : {total_mb:.1f} MB')
    print(f'  Files      : {sum(len(fs) for _, _, fs in files)}')

print('\nBoth indexes saved to Drive ✅')

## Cell 7 — Smoke test: ChromaSARRetriever

Loads the ChromaDB index (instant startup — no re-encoding) and retrieves top-3
structurally similar examples for a sample question.

Compare with Phase 12A Cell 8 (SARRetriever) — results should be identical
but startup is now instant instead of ~30 seconds.

In [ ]:
import sys, time
sys.path.insert(0, '/content/Codegen')

from src.sar.infer import ChromaSARRetriever

start = time.time()
retriever = ChromaSARRetriever(
    model_path='/content/drive/MyDrive/codegen/checkpoints/sar_sql/sar_model.pt',
    chroma_dir=CHROMA_SQL_DIR,
    collection_name='sar_sql',
)
print(f'Startup time: {time.time() - start:.1f} sec  (vs ~30 sec for SARRetriever)\n')

test_question = "How many singers are there in each country?"
results = retriever.retrieve(test_question, top_k=3)

print(f'Query: "{test_question}"')
print('\nTop-3 structurally similar examples from ChromaDB:')
for i, r in enumerate(results, 1):
    print(f'\n  [{i}] Question : {r["question"]}')
    print(f'       SQL      : {r["sql"]}')
    print(f'       Type     : {r["structural_type"]}')